# Brain Tumor MRI Classification — DANN + ResNet-18
### v5 — Fixed & Complete (BrainMRIDataset, Criteria, Full CV Loop)

This notebook implements a **Domain-Adversarial Neural Network (DANN)** using a fine-tuned ResNet-18 for binary brain MRI classification (Tumor vs. No-Tumor).  
It trains on **BRISC-2025** (source domain) while aligning feature distributions with **Mendeley Brain MRI** (unlabelled target domain).

### Fixes in this version
| # | Issue | Fix |
|---|-------|-----|
| 1 | `BrainMRIDataset` class missing | Added in §4.1 with full docstring |
| 2 | `build_transforms()` missing | Added in §4.1 (augmented + val pipelines) |
| 3 | `CLASS_CRITERION` / `DOMAIN_CRITERION` undefined | Defined at top of §7 |
| 4 | Domain soft labels hardcoded to 0/1 | `DOMAIN_SMOOTH_SRC=0.1`, `DOMAIN_SMOOTH_TGT=0.9` |
| 5 | `run_dann_cv()` truncated — no epoch loop | Complete loop: scheduler, scaler, checkpointing, end_fold |
| 6 | `NameError` in `main()` at runtime | All dependencies defined before use |

---
# 1. Environment Setup & Imports <a id='1'></a>

In [3]:
# ==========================================
# 1. Environment Setup & Dependencies
# ==========================================
%pip install kagglehub imagehash 

---
# 2. Unified Imports & Reproducibility
All knobs in one place so you never have to hunt through the code.

In [10]:
# ==========================================
# 2. Unified Imports & Reproducibility
# ==========================================
import os
import re
import sys
import copy
import json
import math
import pickle
import random
import logging
import itertools
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Optional

import cv2
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from PIL import Image
import imagehash
import kagglehub

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve
)

# --- Constants & Seeds ---
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
PIN_MEMORY = torch.cuda.is_available()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# --- Logging ---
out_dir = "/content/Research_Output"
os.makedirs(out_dir, exist_ok=True)
log_file_path = os.path.join(out_dir, "pipeline_history.log")

root_logger = logging.getLogger()
if root_logger.hasHandlers():
    root_logger.handlers.clear()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s — %(message)s",
    handlers=[
        logging.FileHandler(log_file_path, mode='a'), 
        logging.StreamHandler(sys.stdout)
    ]
)
log = logging.getLogger(__name__)
log.info("Pipeline logging successfully initialized and attached to file.")

2026-05-04 17:05:36,290 — Pipeline logging successfully initialized and attached to file.


In [ ]:
# Copy from Drive to local Colab storage
!cp "/content/drive/MyDrive/Mini Project Sem-6(Brain Tumor Classification)/MRI-Dataset.zip" /content/

# Unzip it quietly (-q)
!unzip  -o -q "/content/MRI-Dataset.zip" -d /content/ 

In [ ]:
!find "/content/" -maxdepth 3 -type d

# 3 Data Acquisition & Folder Structuring

In [2]:
# ==========================================
# 3. Data Acquisition & Folder Structuring
# ==========================================
log.info("Downloading datasets via Kaggle and Mendeley...")

# 1. BRISC (Source)
brisc_path = kagglehub.dataset_download("briscdataset/brisc2025")
os.makedirs("/content/DANN/source", exist_ok=True)
os.system(f"cp -r {brisc_path}/brisc2025/classification_task/train /content/DANN/source/")
os.system(f"cp -r {brisc_path}/brisc2025/classification_task/test /content/DANN/source/")

# 2. Mendeley (Target)
os.system("wget -q https://data.mendeley.com/public-api/zip/zwr4ntf94j/download/5 -O target.zip")
os.system("unzip -o -q target.zip -d /content/raw_target/")
os.system("unzip -o -q '/content/raw_target/Brain Tumor MRI Dataset (Glioma, Meningioma, Pitui/Epic and CSCR hospital Dataset.zip' -d '/content/raw_target/'")
os.makedirs("/content/DANN/target", exist_ok=True)
os.system("mv '/content/raw_target/Epic and CSCR hospital Dataset/Train' /content/DANN/target/train 2>/dev/null")
os.system("mv '/content/raw_target/Epic and CSCR hospital Dataset/Test' /content/DANN/target/test 2>/dev/null")

# 3. External Val 1 (Ayesha)
os.system("wget -q https://data.mendeley.com/public-api/zip/w4sw3s9f59/download/1 -O ext1.zip")
os.system("unzip -o -q ext1.zip -d /content/raw_ext1/")
os.system("unzip -o -q '/content/raw_ext1/Brain Tumor Data/Brain Tumor data.zip' -d /content/raw_ext1/")
os.makedirs("/content/External Validation/External-validation-dataset-1", exist_ok=True)
os.system("mv '/content/raw_ext1/Brain Tumor data/Training' '/content/External Validation/External-validation-dataset-1/train' 2>/dev/null")
os.system("mv '/content/raw_ext1/Brain Tumor data/Testing' '/content/External Validation/External-validation-dataset-1/test' 2>/dev/null")

# 4. External Val 2 (Alam)
ext2_path = kagglehub.dataset_download("alamshihab075/brain-tumor-mri-dataset-for-deep-learning")
os.makedirs("/content/External Validation/External-validation-dataset-2", exist_ok=True)
os.system(f"cp -r '{ext2_path}/Train/Train' '/content/External Validation/External-validation-dataset-2/train'")
os.system(f"cp -r '{ext2_path}/test/test' '/content/External Validation/External-validation-dataset-2/test'")

# 5. External Val 3 (DeepPy)
ext3_path = kagglehub.dataset_download("deeppythonist/brain-tumor-mri-dataset")
os.makedirs("/content/External Validation/External-validation-dataset-3", exist_ok=True)
os.system(f"cp -r '{ext3_path}/train' '/content/External Validation/External-validation-dataset-3/train'")
os.system(f"cp -r '{ext3_path}/test' '/content/External Validation/External-validation-dataset-3/test'")

log.info("All datasets acquired and structured.")

Using Colab cache for faster access to the 'brisc2025' dataset.
Using Colab cache for faster access to the 'brain-tumor-mri-dataset-for-deep-learning' dataset.
Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.


# 4. Global Cross-Dataset Deduplication

In [4]:
# ==========================================
# 4. Global Cross-Dataset Deduplication
# ==========================================
ROOT_BASE = Path("/content/")
CLEANED_DATASETS = {
    "Source_BRISC":    {"path": ROOT_BASE / "DANN/source", "priority": 1},
    "Target_Mendeley": {"path": ROOT_BASE / "DANN/target", "priority": 2},
    "Val_Ayesha":      {"path": ROOT_BASE / "External Validation/External-validation-dataset-1", "priority": 3},
    "Val_Alam":        {"path": ROOT_BASE / "External Validation/External-validation-dataset-2", "priority": 4},
    "Val_DeepPy":      {"path": ROOT_BASE / "External Validation/External-validation-dataset-3", "priority": 5},
}

DATASETS_BY_PRIORITY = sorted(CLEANED_DATASETS.items(), key=lambda x: x[1]["priority"])
all_images = []

for name, cfg in DATASETS_BY_PRIORITY:
    if not cfg["path"].exists(): continue
    imgs = [(name, str(p.resolve())) for p in cfg["path"].rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS]
    all_images.extend(imgs)

def compute_phash(filepath):
    try:
        with Image.open(filepath) as img: return str(imagehash.phash(img))
    except Exception: return None

log.info(f"Hashing {len(all_images)} images to hunt for data leaks...")
hash_map = defaultdict(list)

with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {executor.submit(compute_phash, img[1]): img for img in all_images}
    for future in tqdm(as_completed(futures), total=len(all_images), desc="pHashing"):
        dataset_name, filepath = futures[future]
        h = future.result()
        if h: hash_map[h].append((dataset_name, filepath))

internal_pairs, leakage_pairs = [], []
for hash_str, entries in hash_map.items():
    ds_to_paths = defaultdict(list)
    for ds, path in entries: ds_to_paths[ds].append(path)
    
    unique_entries = [] 
    for ds, paths in ds_to_paths.items():
        unique_entries.append((ds, paths[0]))
        for dup_path in paths[1:]: internal_pairs.append((ds, dup_path))
            
    if len(unique_entries) < 2: continue 
    sorted_entries = sorted(unique_entries, key=lambda e: CLEANED_DATASETS[e[0]]["priority"])
    keeper_ds, keeper_path = sorted_entries[0]

    seen_datasets = {keeper_ds}
    for dup_ds, dup_path in sorted_entries[1:]:
        if dup_ds not in seen_datasets:
            leakage_pairs.append((keeper_ds, keeper_path, dup_ds, dup_path))
            seen_datasets.add(dup_ds)

# Execute Purge
deleted_int, deleted_ext = 0, 0
for ds, path in internal_pairs:
    if os.path.exists(path): os.remove(path); deleted_int += 1
for _, _, _, path in leakage_pairs:
    if os.path.exists(path): os.remove(path); deleted_ext += 1

log.info(f"Deduplication Complete. Purged {deleted_int} internal copies and {deleted_ext} cross-dataset leaks.")

pHashing:   0%|          | 0/28815 [00:00<?, ?it/s]

# 5. Configuration & Caching Handlers

In [5]:
# ==========================================
# 5. Configuration & Caching Handlers
# ==========================================
CONFIG = {
    "num_epochs":         25,
    "batch_size":         32,
    "lr":                 1e-4,
    "weight_decay":       1e-4,
    "n_folds":            5,
    "num_workers":        2,
    "seed":               42,
    "grad_clip_norm":     1.0,
    "out_dir":            "/content/Research_Output",
    "brisc_cache_path":    "/content/brisc_cleaned_cache.pkl",
    "mendeley_cache_path": "/content/mendeley_cleaned_cache.pkl",
    "slices_per_patient": 10,
}

_CLAHE = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def preprocess_image(image_path: str, output_size: int = 224) -> np.ndarray | None:
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return None

    # Morphological Crop
    _, thresh = cv2.threshold(img, 10, 255, cv2.THRESH_BINARY)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        all_pts = np.concatenate(contours)
        x, y, w, h = cv2.boundingRect(all_pts)
        img = img[max(0, y - 2): y + h + 2, max(0, x - 2): x + w + 2]

    # Enhance, Rescale, Convert
    img = _CLAHE.apply(img)
    f = img.astype(np.float32)
    lo, hi = f.min(), f.max()
    if hi > lo: f = (f - lo) / (hi - lo)
    img = (f * 255).astype(np.uint8)
    img = cv2.resize(img, (output_size, output_size), interpolation=cv2.INTER_CUBIC)
    return cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

def build_cache(records: list, cache_path: str = "") -> dict:
    if cache_path and os.path.exists(cache_path):
        with open(cache_path, "rb") as f: return pickle.load(f)
    cache = {}
    for r in tqdm(records, desc="Preprocessing to RAM", unit="img"):
        img = preprocess_image(r["path"])
        cache[r["path"]] = img if img is not None else np.zeros((224, 224, 3), dtype=np.uint8)
    if cache_path:
        with open(cache_path, "wb") as f: pickle.dump(cache, f)
    return cache

# 6. Dataset Class & Domain Parsers

In [6]:
# ==========================================
# 6. Dataset Class & Domain Parsers
# ==========================================
BRISC_TYPE_MAP = {"gl": "glioma", "me": "meningioma", "pi": "pituitary", "no": "no_tumor"}
NEGATIVE_FOLDERS = {"no_tumor", "notumor", "no-tumor", "normal", "healthy", "negative"}
TUMOR_KEYWORDS = {"glioma", "meningioma", "pituitary", "tumor", "yes"}
_BRISC_RE = re.compile(r'^brisc2025_(train|test)_(\d{5})_([a-z]{2})_(?:ax|co|sa)_t1', re.IGNORECASE)

def load_brisc(brisc_root: str, slices_per_patient: int = 10) -> list:
    records = []
    for dirpath, _, filenames in os.walk(brisc_root):
        for fname in sorted(filenames):
            if Path(fname).suffix.lower() not in IMG_EXTS: continue
            m = _BRISC_RE.match(Path(fname).stem)
            if not m: continue
            slice_id = int(m.group(2))
            tc = m.group(3).lower()
            patient_id = f"{tc}_{slice_id // slices_per_patient:04d}"
            records.append({"path": os.path.join(dirpath, fname), "label": 0 if tc == "no" else 1, "patient_id": patient_id})
    return records

def load_mendeley(root_path, name: str = "Mendeley") -> list:
    records = []
    for dirpath, _, filenames in os.walk(root_path):
        folder_name = os.path.basename(dirpath).lower()
        if any(neg in folder_name for neg in NEGATIVE_FOLDERS): label = 0
        elif any(pos in folder_name for pos in TUMOR_KEYWORDS): label = 1
        else: continue
        for fname in filenames:
            if Path(fname).suffix.lower() in IMG_EXTS:
                records.append({"path": os.path.join(dirpath, fname), "label": label})
    return records

class BrainMRIDataset(Dataset):
    def __init__(self, records: list, transform=None, cache: dict = None):
        self.records, self.transform, self.cache = records, transform, cache

    def __len__(self): return len(self.records)

    def __getitem__(self, idx: int):
        record = self.records[idx]
        path = record["path"]
        img = self.cache.get(path) if self.cache else preprocess_image(path)
        if img is None: img = np.zeros((224, 224, 3), dtype=np.uint8)
        if self.transform: img = self.transform(img)
        return img, torch.tensor(record["label"], dtype=torch.long)

def build_transforms(augment: bool = False) -> T.Compose:
    imagenet_norm = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    if augment:
        return T.Compose([
            T.ToPILImage(), T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(10),
            T.ColorJitter(brightness=0.2, contrast=0.2), T.RandomAffine(0, translate=(0.05, 0.05)),
            T.ToTensor(), imagenet_norm
        ])
    return T.Compose([T.ToPILImage(), T.ToTensor(), imagenet_norm])

# 7. Model Architecture & Loss Criteria

In [7]:
# ==========================================
# 7. Model Architecture & Loss Criteria
# ==========================================
class GradientReversal(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class DANN_ResNet18(nn.Module):
    def __init__(self, pretrained: bool = True):
        super().__init__()
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.resnet18(weights=weights)
        self.feature_extractor = nn.Sequential(*list(backbone.children())[:-1])
        in_feat = backbone.fc.in_features

        self.class_classifier = nn.Sequential(nn.Dropout(p=0.5), nn.Linear(in_feat, 1))
        self.domain_classifier = nn.Sequential(
            nn.Dropout(p=0.6), nn.Linear(in_feat, 512), nn.BatchNorm1d(512),
            nn.ReLU(True), nn.Dropout(p=0.6), nn.Linear(512, 1)
        )

    def forward(self, x, alpha=None):
        feat = self.feature_extractor(x).view(x.size(0), -1)
        if alpha is not None:
            return self.class_classifier(feat), self.domain_classifier(GradientReversal.apply(feat, alpha))
        return self.class_classifier(feat)

CLASS_CRITERION = nn.BCEWithLogitsLoss()
DOMAIN_CRITERION = nn.BCEWithLogitsLoss()
DOMAIN_SMOOTH_SRC, DOMAIN_SMOOTH_TGT = 0.1, 0.9

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    loss_sum, correct, total, all_labels, all_probs = 0.0, 0, 0, [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.float().unsqueeze(1).to(device)
        with torch.amp.autocast(device_type="cuda", enabled=(device.type == "cuda")):
            logits = model(imgs)
            loss = CLASS_CRITERION(logits, labels)
        probs = torch.sigmoid(logits.float())
        loss_sum += loss.item() * imgs.size(0)
        correct += ((probs >= 0.5).long() == labels.long()).sum().item()
        total += imgs.size(0)
        all_labels.extend(labels.cpu().numpy().flatten())
        all_probs.extend(probs.cpu().numpy().flatten())
    model.train()
    return loss_sum / max(total, 1), correct / max(total, 1), np.array(all_labels), np.array(all_probs)

def compute_metrics(labels, probs):
    preds = (probs >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(labels, preds), "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0), "specificity": tn / (tn + fp + 1e-8),
        "f1": f1_score(labels, preds, zero_division=0), "auc_roc": roc_auc_score(labels, probs) if len(np.unique(labels)) > 1 else 0.0
    }

# 8. Training Runners & External Validation

In [8]:
# ==========================================
# 8. Training Runners & External Validation
# ==========================================
def train_dann_epoch(model, source_loader, target_iter, optimizer, scaler, device, current_epoch, total_epochs, grad_clip_norm):
    model.train()
    loss_sum, correct_class, total = 0.0, 0, 0
    len_dl = len(source_loader)
    
    for i, (src_imgs, src_labels) in enumerate(source_loader):
        tgt_imgs, _ = next(target_iter)
        src_imgs, src_labels = src_imgs.to(device), src_labels.float().unsqueeze(1).to(device)
        tgt_imgs = tgt_imgs.to(device)

        p = float(i + current_epoch * len_dl) / (total_epochs * len_dl)
        alpha = 2.0 / (1.0 + np.exp(-10.0 * p)) - 1.0

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda", enabled=(device.type == "cuda")):
            src_class_logits, src_domain_logits = model(src_imgs, alpha)
            _, tgt_domain_logits = model(tgt_imgs, alpha)
            
            loss_cls = CLASS_CRITERION(src_class_logits, src_labels)
            loss_dom = DOMAIN_CRITERION(src_domain_logits, torch.full_like(src_domain_logits, DOMAIN_SMOOTH_SRC)) + \
                       DOMAIN_CRITERION(tgt_domain_logits, torch.full_like(tgt_domain_logits, DOMAIN_SMOOTH_TGT))
            total_loss = loss_cls + loss_dom

        scaler.scale(total_loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
        scaler.step(optimizer)
        scaler.update()

        n_src = src_imgs.size(0)
        loss_sum += total_loss.item() * n_src
        correct_class += ((torch.sigmoid(src_class_logits) >= 0.5).long() == src_labels.long()).sum().item()
        total += n_src

    return loss_sum / total, correct_class / total

def run_dann_cv(brisc_records, mendeley_records, device, config, brisc_cache=None, mendeley_cache=None):
    labels_arr = np.array([r["label"] for r in brisc_records])
    groups_arr = np.array([r["patient_id"] for r in brisc_records])
    sgkf = StratifiedGroupKFold(n_splits=config["n_folds"], shuffle=True, random_state=config["seed"])
    
    target_ds = BrainMRIDataset(mendeley_records, build_transforms(augment=True), cache=mendeley_cache)
    target_dl = DataLoader(target_ds, batch_size=config["batch_size"], shuffle=True, num_workers=config["num_workers"], drop_last=True, pin_memory=PIN_MEMORY)

    best_global_auc, best_global_model = 0.0, None

    for fold, (tr_idx, vl_idx) in enumerate(sgkf.split(brisc_records, labels_arr, groups=groups_arr), start=1):
        log.info(f"--- Fold {fold} ---")
        train_ds = BrainMRIDataset([brisc_records[i] for i in tr_idx], build_transforms(True), brisc_cache)
        val_ds = BrainMRIDataset([brisc_records[i] for i in vl_idx], build_transforms(False), brisc_cache)
        
        train_dl = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True, num_workers=config["num_workers"], drop_last=True)
        val_dl = DataLoader(val_ds, batch_size=config["batch_size"], shuffle=False)
        target_iter = itertools.cycle(target_dl)

        model = DANN_ResNet18(pretrained=True).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["num_epochs"], eta_min=1e-6)
        scaler = torch.amp.GradScaler(enabled=(device.type == "cuda"))

        best_fold_auc = 0.0
        
        for epoch in range(config["num_epochs"]):
            tr_loss, tr_acc = train_dann_epoch(model, train_dl, target_iter, optimizer, scaler, device, epoch, config["num_epochs"], config["grad_clip_norm"])
            vl_loss, vl_acc, vl_lbl, vl_prb = evaluate(model, val_dl, device)
            scheduler.step()
            
            vl_auc = compute_metrics(vl_lbl, vl_prb)["auc_roc"]
            if vl_auc > best_fold_auc:
                best_fold_auc = vl_auc
                if vl_auc > best_global_auc:
                    best_global_auc, best_global_model = vl_auc, copy.deepcopy(model)
            log.info(f"Ep {epoch+1:02d} | Tr Loss: {tr_loss:.4f} | Vl AUC: {vl_auc:.4f}")
            
    return best_global_model

def run_external_validation(model, datasets, device, config):
    EXT_KEYS = ["Val_Ayesha", "Val_Alam", "Val_DeepPy"]
    results = {}
    for name in EXT_KEYS:
        path = datasets[name]["path"]
        if not path.exists(): continue
        records = load_mendeley(path, name)
        if not records: continue
        
        dl = DataLoader(BrainMRIDataset(records, build_transforms(False)), batch_size=config["batch_size"], num_workers=config["num_workers"])
        _, _, labels, probs = evaluate(model, dl, device)
        results[name] = compute_metrics(labels, probs)
    return results

# 9. Pipeline Ignition

In [9]:
# ==========================================
# 9. Pipeline Ignition
# ==========================================
def main():
    set_seeds(CONFIG["seed"])
    
    brisc_path = str(CLEANED_DATASETS["Source_BRISC"]["path"])
    mendeley_path = str(CLEANED_DATASETS["Target_Mendeley"]["path"])
    
    brisc_records = load_brisc(brisc_path, CONFIG["slices_per_patient"])
    mendeley_records = load_mendeley(mendeley_path)
    
    log.info("Generating Caches...")
    brisc_cache = build_cache(brisc_records, CONFIG["brisc_cache_path"])
    mendeley_cache = build_cache(mendeley_records, CONFIG["mendeley_cache_path"])
    
    log.info("Initiating DANN Cross-Validation...")
    best_model = run_dann_cv(brisc_records, mendeley_records, DEVICE, CONFIG, brisc_cache, mendeley_cache)
    
    if best_model:
        torch.save(best_model.state_dict(), os.path.join(CONFIG["out_dir"], "best_model_global.pth"))
        log.info("Running Strict External Generalization Test...")
        ext_results = run_external_validation(best_model, CLEANED_DATASETS, DEVICE, CONFIG)
        for k, v in ext_results.items(): log.info(f"External [{k}] -> AUC: {v['auc_roc']:.4f} | F1: {v['f1']:.4f}")

if __name__ == "__main__":
    main()

---
# 10. External Validation <a id='10'></a>

Runs the **global best model** (saved after CV) against all three held-out external datasets:

| Dataset | Variable | Clinical relevance |
|---------|----------|--------------------|
| Ayesha et al. | `Val_Ayesha` | Different scanner / centre |
| Alam et al.   | `Val_Alam`   | Different acquisition protocol |
| DeepPy        | `Val_DeepPy` | Curated deep-learning benchmark |

### What you get
- **In the notebook** — a printed metric table (sensitivity, specificity, AUC, F1, accuracy) per dataset, colour-coded in the log, plus an inline 3-panel figure with ROC curves and confusion matrices.
- **In your Drive** (`CONFIG["out_dir"]`) — `external_validation_results.json` (machine-readable) and `external_validation_plots.png` (the same figure, high-res).

> **Why run this separately from CV?**  
> The CV loop uses BRISC folds for train/val — those external datasets were never seen during training *or* hyperparameter selection, making them a true held-out generalisation test.  
> Mixing them into CV would inflate reported performance.


In [21]:
# ==========================================
# 10. External Validation
# ==========================================
# PURPOSE:
#   Evaluate the global best model (output of run_dann_cv) on the 3 external
#   held-out datasets. Results are printed in the notebook AND saved to Drive.
#
# REQUIRES:
#   - best_model  : DANN_ResNet18 instance returned by main() / run_dann_cv()
#   - CLEANED_DATASETS dict from Section 2 (paths already defined)
#   - evaluate() and compute_metrics() from Section 7

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import ConfusionMatrixDisplay, roc_curve


def run_external_validation(
    model:    "DANN_ResNet18",
    datasets: dict,
    device:   "torch.device",
    config:   dict,
) -> dict:
    """
    Evaluates `model` on every dataset in `datasets` and writes results to Drive.

    Args:
        model    : Best global model returned by run_dann_cv().
        datasets : Dict of {name -> Path} — the 3 external val sets from CLEANED_DATASETS.
        device   : torch.device.
        config   : CONFIG dict (uses out_dir, batch_size, num_workers).

    Returns:
        Dict of {dataset_name -> metrics_dict} for downstream use.
    """
    # ── Which datasets to run (skip source + target used during training) ──
    EXTERNAL_KEYS = ["Val_Ayesha", "Val_Alam", "Val_DeepPy"]
    ext_datasets  = {k: datasets[k] for k in EXTERNAL_KEYS if k in datasets}

    if not ext_datasets:
        log.error("No external validation paths found in CLEANED_DATASETS. Check Section 2.")
        return {}

    val_transform = build_transforms(augment=False)   # deterministic only
    all_results   = {}
    all_labels_d  = {}
    all_probs_d   = {}

    log.info("\n" + "═" * 70)
    log.info("  EXTERNAL VALIDATION — Global Best Model")
    log.info("═" * 70)

    for name, path in ext_datasets.items():
        log.info(f"\n  ── {name}  ({path}) ──────────────────────────")

        # ── Load records ───────────────────────────────────────────────────
        records = load_mendeley(root_path=path, name=name)
        if not records:
            log.warning(f"  ⚠  No images found for {name}. Skipping.")
            continue

        # ── No caching for external sets — they run once, RAM is precious ──
        ds = BrainMRIDataset(records, val_transform, cache=None)
        dl = DataLoader(
            ds,
            batch_size=config["batch_size"],
            shuffle=False,
            num_workers=config["num_workers"],
            pin_memory=PIN_MEMORY,
        )

        # ── Evaluate ────────────────────────────────────────────────────────
        _, _, labels, probs = evaluate(model, dl, device)
        metrics = compute_metrics(labels, probs)
        all_results[name]  = metrics
        all_labels_d[name] = labels
        all_probs_d[name]  = probs

        # ── Print metric table inline ───────────────────────────────────────
        log.info(f"  {'Metric':<18}  {'Value':>8}")
        log.info(f"  {'─'*28}")
        for metric, value in metrics.items():
            flag = ""
            if metric == "recall"      and value < 0.80: flag = "  ⚠ LOW SENSITIVITY"
            if metric == "specificity" and value < 0.70: flag = "  ⚠ LOW SPECIFICITY"
            if metric == "auc_roc"     and value < 0.75: flag = "  ⚠ LOW AUC"
            log.info(f"  {metric:<18}  {value:>8.4f}{flag}")
        log.info(f"  {'─'*28}")
        log.info(f"  n_images = {len(records)},  n_tumor = {sum(r['label'] for r in records)},  n_no_tumor = {len(records) - sum(r['label'] for r in records)}")

    # ── Save JSON to Drive ──────────────────────────────────────────────────
    json_out = os.path.join(config["out_dir"], "external_validation_results.json")
    with open(json_out, "w") as f:
        json.dump(all_results, f, indent=2)
    log.info(f"\n  ✅ Results saved → {json_out}")

    # ── Plot: ROC curves + Confusion Matrices ──────────────────────────────
    n = len(all_results)
    if n == 0:
        log.warning("No results to plot.")
        return all_results

    fig = plt.figure(figsize=(7 * n, 10))
    fig.suptitle("External Validation — Global Best DANN Model", fontsize=16, fontweight="bold", y=1.01)
    gs  = gridspec.GridSpec(2, n, figure=fig, hspace=0.4, wspace=0.35)

    for col, (name, metrics) in enumerate(all_results.items()):
        labels = all_labels_d[name]
        probs  = all_probs_d[name]
        preds  = (probs >= 0.5).astype(int)

        # ── ROC curve (top row) ────────────────────────────────────────────
        ax_roc = fig.add_subplot(gs[0, col])
        fpr, tpr, _ = roc_curve(labels, probs)
        ax_roc.plot(fpr, tpr, lw=2, color="#4C9BE8", label=f"AUC = {metrics['auc_roc']:.3f}")
        ax_roc.plot([0, 1], [0, 1], "k--", lw=1)
        ax_roc.set_xlim([0, 1]); ax_roc.set_ylim([0, 1.02])
        ax_roc.set_xlabel("False Positive Rate"); ax_roc.set_ylabel("True Positive Rate")
        ax_roc.set_title(f"{name}\nROC Curve", fontsize=11)
        ax_roc.legend(loc="lower right", fontsize=10)
        # Annotate key metrics below the title
        ax_roc.text(0.05, 0.10,
            f"Sens={metrics['recall']:.3f}  Spec={metrics['specificity']:.3f}\n"
            f"F1={metrics['f1']:.3f}  Acc={metrics['accuracy']:.3f}",
            transform=ax_roc.transAxes, fontsize=9,
            bbox=dict(boxstyle="round,pad=0.3", facecolor="#F0F4FF", edgecolor="#4C9BE8"))

        # ── Confusion matrix (bottom row) ──────────────────────────────────
        ax_cm = fig.add_subplot(gs[1, col])
        ConfusionMatrixDisplay.from_predictions(
            labels, preds,
            labels=[0, 1],  # <--- ADD THIS EXACT LINE HERE
            display_labels=["No Tumor", "Tumor"],
            colorbar=False,
            ax=ax_cm,
            cmap="Blues",
        )
        ax_cm.set_title(f"{name}\nConfusion Matrix", fontsize=11)

    plt.tight_layout()

    # ── Save figure to Drive ───────────────────────────────────────────────
    plot_out = os.path.join(config["out_dir"], "external_validation_plots.png")
    plt.savefig(plot_out, dpi=150, bbox_inches="tight")
    log.info(f"  ✅ Plot saved     → {plot_out}")

    # ── Show inline in notebook ────────────────────────────────────────────
    plt.show()

    return all_results


# ── CROSS-VALIDATION SUMMARY TABLE ────────────────────────────────────────
def print_external_summary_table(ext_results: dict):
    """Prints a clean comparison table across all external datasets."""
    if not ext_results:
        return
    metrics_order = ["accuracy", "precision", "recall", "specificity", "f1", "auc_roc"]
    header = f"  {'Dataset':<18}" + "".join(f"  {m:>12}" for m in metrics_order)
    log.info("\n" + "═" * len(header))
    log.info("  EXTERNAL VALIDATION — SUMMARY TABLE")
    log.info("═" * len(header))
    log.info(header)
    log.info("  " + "─" * (len(header) - 2))
    for name, metrics in ext_results.items():
        row = f"  {name:<18}" + "".join(f"  {metrics.get(m, 0.0):>12.4f}" for m in metrics_order)
        log.info(row)
    log.info("═" * len(header))


In [22]:

# 1. Instantiate the model architecture (no need to download pretrained ImageNet weights again)
saved_model = DANN_ResNet18(pretrained=False).to(DEVICE)

# 2. Load the weights you already spent hours training from your Google Drive
model_path = os.path.join(CONFIG["out_dir"], "best_model_global.pth")
saved_model.load_state_dict(torch.load(model_path))
log.info(f"Successfully loaded trained model from {model_path}")

# 3. Run ONLY the external validation step
log.info("\n>>> Starting External Validation (Inference Only) <<<")
ext_results = run_external_validation(
    model=saved_model,
    datasets=CLEANED_DATASETS,
    device=DEVICE,
    config=CONFIG,
)

# 4. Print the final summary table
print_external_summary_table(ext_results)

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1179: UndefinedMetricWarning: No negative samples in y_true, false positive value should be meaningless
  warnings.warn(
/tmp/ipykernel_3292/3304643675.py:140: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


In [4]:
import os
from pathlib import Path

def get_detailed_counts(dataset_path):
    root = Path(dataset_path)
    if not root.exists():
        print(f"❌ Path not found: {root}")
        return

    print(f"\n{'='*50}")
    print(f"📊 Dataset: {root.name}")
    print(f"{'='*50}")

    # Dictionary to store counts: split -> class -> count
    counts = {}
    img_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

    for dirpath, _, filenames in os.walk(root):
        # Count only image files
        images = [f for f in filenames if Path(f).suffix.lower() in img_exts]
        if not images:
            continue
            
        # Determine the subfolder structure (e.g., 'train/glioma_tumor')
        rel_path = Path(dirpath).relative_to(root)
        parts = rel_path.parts
        
        if len(parts) >= 2:
            split_name = parts[0] # Usually 'train', 'test', 'Training', or 'Testing'
            class_name = parts[1] # Usually 'glioma', 'pituitary', etc.
        elif len(parts) == 1:
            split_name = "mixed_split"
            class_name = parts[0]
        else:
            split_name = "root"
            class_name = "root"

        if split_name not in counts:
            counts[split_name] = {}
        if class_name not in counts[split_name]:
            counts[split_name][class_name] = 0
            
        counts[split_name][class_name] += len(images)

    # Print the formatted results
    if not counts:
        print("  [Empty Directory or No Images Found]")
        return

    for split, classes in sorted(counts.items()):
        print(f"\n📂 Split: [{split.upper()}]")
        total_in_split = 0
        for cls_name, count in sorted(classes.items()):
            print(f"   ├── {cls_name:<18} : {count}")
            total_in_split += count
        print(f"   └── {'TOTAL':<18} : {total_in_split}")

# Run this on your 3 external validation directories
get_detailed_counts(CLEANED_DATASETS["Source_BRISC"])
get_detailed_counts(CLEANED_DATASETS["Target_Mendeley"])
get_detailed_counts(CLEANED_DATASETS["Val_Ayesha"])
get_detailed_counts(CLEANED_DATASETS["Val_Alam"])
get_detailed_counts(CLEANED_DATASETS["Val_DeepPy"])


📊 Dataset: source

📂 Split: [TEST]
   ├── glioma             : 253
   ├── meningioma         : 294
   ├── no_tumor           : 138
   ├── pituitary          : 300
   └── TOTAL              : 985

📂 Split: [TRAIN]
   ├── glioma             : 1144
   ├── meningioma         : 1212
   ├── no_tumor           : 1056
   ├── pituitary          : 1433
   └── TOTAL              : 4845

📊 Dataset: target

📂 Split: [TEST]
   ├── glioma             : 307
   ├── meningioma         : 227
   ├── notumor            : 368
   ├── pituitary          : 42
   └── TOTAL              : 944

📂 Split: [TRAIN]
   ├── glioma             : 1372
   ├── meningioma         : 296
   ├── notumor            : 813
   ├── pituitary          : 666
   └── TOTAL              : 3147

📊 Dataset: External-validation-dataset-1

📂 Split: [TEST]
   ├── notumor            : 11
   └── TOTAL              : 11

📂 Split: [TRAIN]
   ├── glioma             : 39
   ├── meningioma         : 1
   ├── notumor            : 259
   └── TOTAL  